# Airline Feature Engineering v2(1)

Generate the 25 binary features and evidence diagnostics from the cleaned dataset. Historical threshold (>6h) is preserved to reproduce existing classifications.

**Execution:** Start Jupyter inside the project tree. Run the path setup first. Review configuration and existing outputs before executing.

In [ ]:
# Resolve the project from the notebook working directory, not a machine-specific path.
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if all((p / n).is_dir() for n in ("Python Code", "Data", "Streamlit"))), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from inside the AEGEAN project tree.")
SCRAPING_DIR = PROJECT_ROOT / "Data" / "Scraping"
CLEANING_DIR = PROJECT_ROOT / "Data" / "Output" / "Cleaning data"
HF_DIR = PROJECT_ROOT / "Data" / "Output" / "Hugging Face"
LLM_OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output" / "LLM"
for directory in (CLEANING_DIR, HF_DIR, LLM_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


# Airline reviews: 25-feature engineering (corrected competition groups)
Run all cells in order. No sentiment analysis is performed.
Regional: Aegean Airlines, Bangkok Air, Porter Airlines, Virgin Australia.
European: Aegean Airlines, Air France, Lufthansa, Turkish Airlines.


In [ ]:
%pip install pandas numpy tqdm airportsdata scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.insert(0, str(PROJECT_ROOT / "Python Code" / "Hugging Face"))
from airline_features_v2 import ROOT, FEATURES, REGIONAL, EUROPEAN, run_pipeline, run_tests, evaluate_manual_labels
print("Features:",len(FEATURES))
print("Regional:",REGIONAL,"\nEuropean:",EUROPEAN)

In [ ]:
if "df_final" not in globals() or not isinstance(df_final,pd.DataFrame):
    df_final = pd.read_csv(CLEANING_DIR / "df_final.csv", encoding="utf-8-sig", low_memory=False)
print(df_final.shape)
print(df_final["airline"].value_counts())

In [ ]:
df_final["airline"] = df_final["airline"].replace(to_replace="aegean_airlines", value="Aegean Airlines")
print(df_final["airline"].value_counts())

In [ ]:
test_errors = run_tests()
assert test_errors.empty, test_errors.to_string(index=False)
print("Existing regression tests passed")

In [ ]:
(df_final_features, df_final_features_filtered, df_classification_diagnostics, feature_summary, feature_summary_by_airline, feature_summary_by_source, df_validation, classification_conflicts) = run_pipeline(df_final, output_dir=ROOT, save=True)
print("Full:",df_final_features.shape,"Filtered:",df_final_features_filtered.shape)
display(feature_summary)

In [ ]:
display(df_final_features.groupby(["regional","european"])["airline"].value_counts())
display(feature_summary_by_airline)
display(classification_conflicts.head(20))

In [ ]:
# After manually filling the manual_label column in classification_validation_sample.csv:
labeled = pd.read_csv(ROOT / "classification_validation_sample.csv")
display(evaluate_manual_labels(labeled))